# RT-DETR 모델 업로드 평가 (COCO val2017)
FP32와 양자화한 **ONNX 모델 파일 자체**를 올려 같은 COCO 평가 이미지 1,000장을 추론합니다. mAP·클래스별 AP, 실제 ONNX 파일 크기, 프로세스 RAM, 추론 지연·FPS를 기록하고 결과를 ZIP으로 내려받습니다. CUDA에서는 가능할 때 프로세스 GPU 메모리도 기록합니다. 모델 파일 이름만으로 INT8/INT4를 인정하지 않고 그래프 증거를 별도로 기록합니다.

모델은 [공식 RT-DETR deploy ONNX 인터페이스](https://github.com/lyuwenyu/RT-DETR/blob/main/rtdetr_pytorch/tools/export_onnx.py)인 `images`, `orig_target_sizes` 입력과 `labels`, `boxes`, `scores` 출력을 사용해야 합니다. `.pth` 파일은 직접 평가할 수 없습니다.

## 1. 실행 환경과 데이터 준비
가능하면 Colab의 GPU 런타임을 선택하세요. 아래 `provider`를 `cuda` 또는 `cpu`로 정하고 실행합니다. FP32·W8A8·W4A16 비교는 반드시 **같은 런타임과 provider**에서 하세요. COCO 다운로드는 약 1.1GB입니다.

In [ ]:
provider = 'cuda'  # CPU 런타임이면 'cpu'로 변경
import os, pathlib, subprocess, sys
repo = pathlib.Path('/content/haiteam4_finetune')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/foxyaef/haiteam4_finetune.git', str(repo)], check=True)
os.chdir(repo)
requirements = 'requirements-eval-gpu.txt' if provider == 'cuda' else 'requirements-eval-cpu.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', requirements], check=True)
subprocess.run([sys.executable, 'prepare_data.py'], check=True)
print('준비 완료:', repo)

## 2. 모델 업로드
FP32 기준 모델과 비교할 양자화 모델을 한 번에 업로드합니다. 예: `fp32.onnx`, `backbone_w8a8.onnx`, `backbone_w4a16.onnx`. 기준 모델을 `fp32.onnx`로 이름을 맞추면 차이값이 자동 계산됩니다. 대용량 파일이면 Google Drive 경로를 사용할 수 있습니다. 각 모델은 **단일 self-contained `.onnx`** 파일이어야 합니다.

In [ ]:
from google.colab import files
model_dir = repo / 'uploaded_models'
model_dir.mkdir(exist_ok=True)
os.chdir(model_dir)
uploaded = files.upload()  # FP32 + 양자화 모델 여러 개 선택
os.chdir(repo)
model_paths = sorted(model_dir / name for name in uploaded if name.lower().endswith('.onnx'))
del uploaded
if not model_paths:
    raise ValueError('ONNX 모델을 업로드하세요')
print('평가할 모델:', [p.name for p in model_paths])
# 대용량 모델을 Drive에서 사용할 때:
# from google.colab import drive
# drive.mount('/content/drive')
# model_paths = sorted(pathlib.Path('/content/drive/MyDrive/rtdetr_models').glob('*.onnx'))

## 3. 실제 이미지 추론·정확도·자원 측정
각 모델을 별도 프로세스에서 같은 1,000장으로 평가합니다. 첫 10장은 워밍업 추론도 실행하고 지연 통계에서는 제외합니다. 첫 세 이미지의 탐지 결과를 미리보기로 저장합니다. 현재 GPU/ONNX Runtime이 어떤 모델 연산을 지원하지 않으면 그 모델에서 오류가 납니다.

In [ ]:
import datetime, json, csv, shutil
from IPython.display import display, Image as DisplayImage
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')
run_dir = repo / 'results' / ('models_' + stamp)
run_dir.mkdir(parents=True)
summary = []
for model in model_paths:
    label = model.stem
    output = run_dir / label
    print('\n평가 중:', model.name, flush=True)
    subprocess.run([sys.executable, 'scripts/evaluate_model.py', '--model', str(model),
                    '--output', str(output), '--provider', provider], cwd=repo, check=True)
    metrics = json.loads((output / 'metrics.json').read_text(encoding='utf-8'))
    resource = metrics['resource']
    summary.append({'model': label, 'mAP50_95': metrics['mAP50_95'],
                    'AP50': metrics['mAP50'], 'AP75': metrics['mAP75'],
                    'file_MiB': resource['model_file_mib'],
                    'peak_RSS_MiB': resource['rss_peak_bytes'] / 1048576,
                    'RSS_increase_MiB': resource['rss_increase_bytes'] / 1048576,
                    'GPU_process_peak_MiB': (resource['gpu_process_peak_bytes'] / 1048576
                        if resource['gpu_process_peak_bytes'] is not None else None),
                    'latency_p50_ms': resource['inference_ms_p50'],
                    'latency_p95_ms': resource['inference_ms_p95'],
                    'FPS': resource['inference_fps']})
baseline = next((row for row in summary if row['model'] == 'fp32'), None)
for row in summary:
    row['delta_mAP50_95'] = row['mAP50_95'] - baseline['mAP50_95'] if baseline else None
    row['size_vs_fp32'] = row['file_MiB'] / baseline['file_MiB'] if baseline else None
with (run_dir / 'summary.csv').open('w', newline='', encoding='utf-8-sig') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(summary[0]))
    writer.writeheader()
    writer.writerows(summary)
display(summary)
for model in model_paths:
    preview = run_dir / model.stem / 'preview_1.jpg'
    if preview.exists():
        print(model.stem, '미리보기')
        display(DisplayImage(filename=str(preview)))
bundle = shutil.make_archive(str(run_dir), 'zip', root_dir=run_dir)
files.download(bundle)